In [37]:
import overpy
import pandas as pd 
import requests
import json
import pymssql
from sqlalchemy import Integer, String, Float, DATETIME, create_engine

In [38]:
# Load configuration from config/db_config.json
with open('../config/db_config.json', 'r') as f:
    db_config = json.load(f)

# Get database credentials
server = db_config['server']
database = db_config['database']
db_user = db_config['db_user']
db_password = db_config['db_password']# Connect to SQL Database
conn = pymssql.connect(server, db_user, db_password, database)

# Create connection string for SQLAlchemy
connection_string = f"mssql+pymssql://{db_user}:{db_password}@{server}/{database}"
engine = create_engine(connection_string)

In [39]:
# SQL-Abfrage für die gewünschten Spalten
query = "SELECT DISTINCT id FROM OVRP_HikingRoutes"

# Daten aus der Azure SQL-Datenbank laden
df_hikingroute_ids = pd.read_sql_query(query, con=engine)

# IDs direkt in eine Liste umwandeln
list_hikingroute_ids = df_hikingroute_ids['id'].tolist()

# Ergebnis anzeigen
print(list_hikingroute_ids)

[120125, 121950, 121951, 121952, 121955, 121956, 121958, 135975, 149889, 151145, 151146, 151147, 153094, 157491, 157492, 157493, 157925, 162056, 166101, 167394, 167853, 168320, 168421, 169179, 169199, 169200, 169201, 169202, 169203, 169204, 169206, 169997, 170847, 170849, 170850, 170851, 172080, 172131, 172132, 172223, 172224, 172823, 172824, 172825, 172826, 172828, 175309, 176643, 176645, 176646, 176648, 180156, 182544, 182545, 182549, 182550, 187023, 187024, 187025, 187026, 187027, 188401, 188402, 188403, 188404, 190837, 192242, 192632, 192633, 192638, 192948, 192959, 192960, 192962, 192963, 192964, 214477, 233163, 233165, 233166, 233167, 251677, 251743, 252798, 252800, 252801, 252802, 252803, 271157, 272253, 272254, 272256, 272257, 273000, 274466, 274467, 274468, 274489, 274490, 274491, 275400, 275401, 275403, 275404, 275406, 276094, 276095, 276096, 279487, 279490, 279491, 280483, 280484, 283632, 285445, 285446, 285447, 285448, 285449, 285450, 285458, 285577, 285578, 285579, 285580,

In [40]:
# Verbindung zur Overpass-API
api = overpy.Overpass(url="http://overpass.osm.ch/api/interpreter")

# Zeitstempel für die API-Aufrufe
timestamp_apicall = pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S")


# Leere Liste für die Ergebnisse
results = []

# Overpass-Abfragen iterativ durchführen
for route_id in list_hikingroute_ids:
    # Overpass-Abfrage für die aktuelle Relation
    query = f"""
    [out:json];
    relation["route"="hiking"](id:{route_id});
    way(r);
    out ids center tags;
    """
    try:
        # Anfrage an die Overpass-API senden
        result = api.query(query)

        # Iteration durch alle Wege, die zur Relation gehören
        for way in result.ways:
            if way.center_lat and way.center_lon:
                # Ergebnisse als Dictionary speichern
                results.append({
                    "id": way.id,
                    "itshikingroute": route_id,
                    "lat": way.center_lat,
                    "lon": way.center_lon,
                    "timestamp_apicall": timestamp_apicall
                })

    except overpy.exception.OverpassException as e:
        print(f"Fehler bei der Anfrage für ID {route_id}: {e}")

# Ergebnisse in einen DataFrame umwandeln
df_waypoints = pd.DataFrame(results)

# Daten bereinigen und konvertieren
df_waypoints['lat'] = pd.to_numeric(df_waypoints['lat'], errors='coerce')
df_waypoints['lon'] = pd.to_numeric(df_waypoints['lon'], errors='coerce')
df_waypoints['timestamp_apicall'] = pd.to_datetime(df_waypoints['timestamp_apicall'], errors='coerce')

# Ergebnis anzeigen
print(df_waypoints.head())

         id  itshikingroute        lat       lon   timestamp_apicall
0  28022787          120125  46.871293  6.581740 2024-11-23 14:25:36
1  28022788          120125  46.874111  6.589236 2024-11-23 14:25:36
2  28022789          120125  46.878365  6.598630 2024-11-23 14:25:36
3  33636662          120125  46.867863  6.573559 2024-11-23 14:25:36
4  36789486          120125  46.880193  6.612009 2024-11-23 14:25:36


In [41]:
# Create table if it doesn't exist
query = f"""
        CREATE TABLE OVRP_waypoints (
            id                      INT         NOT NULL PRIMARY KEY,
            itshikingroute          INT         NOT NULL,
            lat                     FLOAT       NOT NULL,
            lon                     FLOAT       NOT NULL,
            timestamp_apicall       DATETIME    NULL,
            FOREIGN KEY (itshikingroute) REFERENCES OVRP_HikingRoutes (id)
        );
    """

conn = pymssql.connect(server, db_user, db_password, database)
cursor = conn.cursor()
cursor.execute(query)

conn.commit()
conn.close()

In [42]:
table_name = "OVRP_waypoints"

# Create connection string for SQLAlchemy
connection_string = f"mssql+pymssql://{db_user}:{db_password}@{server}/{database}"
engine = create_engine(connection_string)

# Ingest data to tabledatabase table
df_waypoints.to_sql(table_name, con=engine, if_exists='replace', index=False)
print("DataFrame erfolgreich in die MSSQL-Datenbank geladen!")

DataFrame erfolgreich in die MSSQL-Datenbank geladen!
